# Mercado de restaurantes de Los Angeles
## Análise de oportunidade para uma rede de cafetarias com serviço robotizado

**Resumo para decisão de investidores** · outubro de 2026

☕🤖 Esta pesquisa avalia se a oferta existente sustenta um piloto de cafetaria robotizada e que evidências seriam necessárias antes de investir numa expansão. Os dados descrevem estabelecimentos e lugares; não medem vendas, procura local nem retorno dos robôs.


**#### Apresentação em PDF**

Apresentação site estático: <[https://la-restaurant-market-analysis.onrender.com](https://la-restaurant-market-analysis.onrender.com)>
Documentação para análise: <[https://github.com/carolacezalves/la-restaurant-market-analysis/tree/main](https://github.com/carolacezalves/la-restaurant-market-analysis/tree/main)>


### Índice
1. Qualidade e âmbito dos dados
2. Composição do mercado e presença de redes
3. Comparáveis de cafetarias de rede
4. Capacidade de lugares e estrutura das redes
5. Verificação do paradoxo de Simpson
6. Localização e capacidade por rua
7. Verificação de interesse de pesquisa no Wordstat
8. Síntese para o investidor
9. Conclusão e próximos passos

### Hipóteses para testar
- **☕ Espaço de mercado:** a categoria café tem escala suficiente e sinais de operação replicável?
- **🪑 Formato:** os comparáveis apontam para uma capacidade típica ou para formatos muito diversos?
- **📍 Localização:** concentração de restaurantes ajuda a encontrar procura ou também sinaliza concorrência intensa?
- **🤖 Recorrência:** que resultados do piloto demonstrariam que a proposta continua a atrair clientes depois da novidade?

Cada secção responde a uma destas perguntas. As notas após os gráficos distinguem o que os dados mostram do que deve ser validado antes de comprometer capital.

## 1. Base de análise e confiança dos dados

A análise cobre **9 651 estabelecimentos**. As verificações identificaram **zero linhas duplicadas**, **zero contagens de lugares inválidas** e apenas **três classificações de rede desconhecidas**. Estes resultados permitem comparar a oferta com confiança; os três registos sem classificação ficam fora das comparações entre redes e independentes.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import display, Markdown

DATA_PATH = Path('/datasets/rest_data_us.csv')
if not DATA_PATH.exists():
    # Search beside the notebook and in the repository's standard data folder.
    candidates = [
        Path.cwd() / 'rest_data_us.csv',
        Path.cwd() / 'data' / 'rest_data_us.csv',
        Path.cwd().parent / 'rest_data_us.csv',
        Path.cwd().parent / 'data' / 'rest_data_us.csv',
    ]
    DATA_PATH = next((path for path in candidates if path.exists()), None)
    if DATA_PATH is None:
        raise FileNotFoundError('Put rest_data_us.csv beside the notebook or in the repository data/ folder, then update DATA_PATH.')

raw = pd.read_csv(DATA_PATH)
print(f'Dados carregados: {len(raw):,} linhas × {raw.shape[1]} colunas, a partir de {DATA_PATH}')
display(raw.head())
print('Colunas:', raw.columns.tolist())

In [ ]:
# Padronizar cabeçalhos e converter os campos para tipos analíticos explícitos.
df = raw.copy()
df.columns = df.columns.str.strip().str.lower()
expected = {'id', 'object_name', 'address', 'chain', 'object_type', 'number'}
missing_columns = expected - set(df.columns)
if missing_columns:
    raise ValueError(f'Required columns missing: {sorted(missing_columns)}')

df['id'] = pd.to_numeric(df['id'], errors='coerce').astype('Int64')
df['number'] = pd.to_numeric(df['number'], errors='coerce')
df['object_name'] = df['object_name'].astype('string').str.strip()
df['address'] = df['address'].astype('string').str.strip()
df['object_type'] = df['object_type'].astype('string').str.strip().str.title()
df['object_type'] = df['object_type'].replace({'Restaurant':'Restaurante', 'Fast Food':'Fast food', 'Cafe':'Café', 'Pizza':'Pizzaria', 'Bar':'Bar', 'Bakery':'Padaria'})
chain_map = {'true': True, 'false': False, '1': True, '0': False}
df['chain'] = df['chain'].astype('string').str.strip().str.lower().map(chain_map).astype('boolean')

quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique': df.nunique(dropna=True)
})
display(quality)
print(f'IDs duplicados: {df.id.duplicated().sum():,}')
print(f'Linhas totalmente duplicadas: {df.duplicated().sum():,}')
print(f'Números de lugares inválidos ou não positivos: {(df.number.isna() | (df.number <= 0)).sum():,}')
print(f'Estabelecimentos sem classificação de rede após normalização: {df.chain.isna().sum():,}')

### Critérios de comparação

As classificações desconhecidas mantêm-se separadas dos estabelecimentos independentes. As comparações de capacidade usam apenas registos com contagem de lugares válida. Assim, as diferenças apresentadas refletem a oferta observada e não uma estimativa de vendas ou de desempenho financeiro.

In [ ]:
# Resolver IDs duplicados de forma conservadora: conservar o primeiro registo e contabilizar a ocorrência.
duplicate_id_rows = int(df.id.duplicated(keep='first').sum())
df = df.drop_duplicates(subset='id', keep='first').copy()

# Manter as classificações de rede desconhecidas nas análises gerais de estabelecimentos.
valid_type = df[df.object_type.notna() & df.object_type.ne('')].copy()
valid_seats = valid_type[valid_type.number.notna() & df.number.gt(0)].copy()
classified = valid_type[valid_type.chain.notna()].copy()

print(f'Linhas para análise após remoção de IDs duplicados: {len(df):,}')
print(f'Tipos válidos: {len(valid_type):,}; registos com lugares válidos: {len(valid_seats):,}')
print(f'Classificação de rede conhecida: {len(classified):,}; desconhecida: {df.chain.isna().sum():,}')
print(f'Linhas removidas por ID duplicado: {duplicate_id_rows:,}')

## 2. Composição do mercado

### 🍽️ Tipos de estabelecimento
O peso de cada categoria indica a composição do inventário. Uma categoria numerosa representa concorrência estabelecida, mas não prova maior procura nem maior potencial de receita.

**Interatividade:** passe o cursor sobre as fatias para consultar os valores. Os gráficos interativos usam Plotly; se necessário, execute `%pip install plotly` numa célula e reinicie o kernel.

In [ ]:
sns.set_theme(style='whitegrid', palette='deep', context='notebook')
plt.rcParams['font.family'] = ['DejaVu Sans']  # Compatível com ambientes Linux/Jupyter.
type_counts = valid_type.object_type.value_counts().rename_axis('type').reset_index(name='venues')
type_counts['share_pct'] = 100 * type_counts.venues / type_counts.venues.sum()
fig = px.pie(type_counts, names='type', values='venues', hole=.52,
             custom_data=['share_pct'], color_discrete_sequence=px.colors.qualitative.Safe,
             title='Composição dos estabelecimentos por tipo')
fig.update_traces(textposition='inside', textinfo='percent', textfont_size=15,
                  hovertemplate='<b>%{label}</b><br>Estabelecimentos: %{value:,}<br>Percentagem: %{customdata[0]:.1f}%<extra></extra>')
fig.update_layout(
    template='plotly_white', width=1180, height=720,
    margin=dict(l=35,r=280,t=95,b=35),
    legend=dict(orientation='v', x=1.02, xanchor='left', y=.5, yanchor='middle',
                title=dict(text='Tipo de estabelecimento', font=dict(size=16)),
                font=dict(size=15), itemsizing='constant', tracegroupgap=8),
    uniformtext_minsize=12, uniformtext_mode='hide'
)
fig.add_annotation(text=f"<b>{type_counts.venues.sum():,}</b><br>estabelecimentos",
                   x=.5,y=.5,showarrow=False,font=dict(size=17,color='#34495E'))
fig.show()
display(type_counts)

In [ ]:
top_type = type_counts.iloc[0]
cafe_share = type_counts.loc[type_counts.type.eq('Café'), 'share_pct'].squeeze()
display(Markdown(f"**Leitura para a decisão.** {top_type['type']} representa {top_type.share_pct:.1f}% da oferta listada; as cafés representam {cafe_share:.1f}%. A menor dimensão relativa da categoria café pode justificar testar posicionamento e concorrência local, mas não demonstra procura por uma cafetaria com robôs. Antes de investir, validar fluxo de clientes, preços e preferência pelo serviço no corredor escolhido."))

### 🔗 Redes e estabelecimentos independentes
A comparação responde a duas perguntas distintas: que tipos concentram mais unidades de rede e, dentro de cada categoria, que proporção pertence a uma rede. Esta distinção ajuda a avaliar a existência de formatos replicáveis sem confundir escala com rentabilidade.

In [ ]:
chain_counts = classified.chain.value_counts().rename(index={True:'Rede', False:'Independente'}).rename_axis('status').reset_index(name='venues')
chain_counts['share_pct'] = chain_counts.venues / chain_counts.venues.sum() * 100
fig = px.pie(chain_counts, names='status', values='venues', hole=.58,
             color='status', color_discrete_map={'Independente':'#197C78','Rede':'#D98841'},
             title='Estabelecimentos independentes e pertencentes a redes')
fig.update_traces(textposition='inside', textinfo='percent', textfont_size=18,
                  hovertemplate='<b>%{label}</b><br>Estabelecimentos: %{value:,}<br>Percentagem: %{percent}<extra></extra>')
fig.update_layout(
    template='plotly_white', width=920, height=640,
    margin=dict(l=45,r=45,t=95,b=120),
    legend=dict(orientation='h', x=.5, xanchor='center', y=-.13, yanchor='top',
                title=dict(text='Classificação', font=dict(size=16)),
                font=dict(size=17), itemsizing='constant', tracegroupgap=18),
    uniformtext_minsize=14, uniformtext_mode='hide'
)
fig.add_annotation(text=f"<b>{chain_counts.venues.sum():,}</b><br>classificados",
                   x=.5,y=.5,showarrow=False,font=dict(size=18,color='#34495E'))
fig.show()
display(chain_counts)

In [ ]:
type_chain = classified.groupby(['object_type','chain']).size().rename('venues').reset_index()
type_chain['share_within_type_pct'] = type_chain.venues / type_chain.groupby('object_type').venues.transform('sum') * 100
type_order = type_chain.groupby('object_type').venues.sum().sort_values().index
type_chain['object_type'] = pd.Categorical(type_chain['object_type'], categories=type_order, ordered=True)
fig, ax = plt.subplots(figsize=(10,5.5))
sns.scatterplot(data=type_chain, x='share_within_type_pct', y='object_type', hue='chain',
                s=115, hue_order=[False,True], palette={False:'#197C78',True:'#D98841'},
                legend=False, ax=ax)
for _, row in type_chain.iterrows():
    ax.annotate(f"{row.venues:,}", (row.share_within_type_pct, row.object_type),
                xytext=(7,0), textcoords='offset points', va='center', fontsize=8)
ax.set(title='Participação de redes dentro de cada tipo', xlabel='Estabelecimentos classificados como rede (%)', ylabel='')
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0],marker='o',color='w',label='Independente',markerfacecolor='#197C78',markersize=9),
                   Line2D([0],[0],marker='o',color='w',label='Rede',markerfacecolor='#D98841',markersize=9)],
          title='Classificação', loc='lower right')
sns.despine(); plt.tight_layout(); plt.show()
display(type_chain.pivot(index='object_type',columns='chain',values='share_within_type_pct').rename(columns={False:'Independentes (%)',True:'Redes (%)'}).round(1))

In [ ]:
chain_by_type = type_chain[type_chain.chain].sort_values('venues', ascending=False)
most_common_chain_type = chain_by_type.iloc[0]
cafe_chain_share = type_chain.loc[(type_chain.object_type == 'Café') & type_chain.chain, 'share_within_type_pct'].iloc[0]
display(Markdown(f"**Leitura para a decisão.** Os restaurantes representam o maior número absoluto de unidades de rede ({most_common_chain_type.venues:,}), enquanto {cafe_chain_share:.1f}% das cafés com classificação conhecida pertencem a redes. Há precedentes para operar formatos replicáveis também na categoria café; isso apoia estudar um piloto padronizável, mas não comprova a viabilidade de uma rede de cafetarias robotizadas."))

## 3. Comparáveis de cafetarias de rede ☕

Para não confundir uma palavra no nome com uma categoria comercial, a análise separa dois sinais: **comparáveis principais** (Chain = True e tipo Café) e **nomes para revisão** (Chain = True e termos como coffee, café, espresso, tea ou boba no nome/tipo, mas classificados noutro tipo). Só os comparáveis principais definem a referência de capacidade; os casos de nome são pistas para verificação, não confirmações de que sejam cafetarias.

In [ ]:
# Reutilizar uma única definição em toda a comparação de cafetarias de rede.
chain_true = valid_seats['chain'].eq(True).fillna(False)
cafe_type = valid_seats['object_type'].eq('Café')
cafe_network_mask = chain_true & cafe_type
name_type = valid_seats['object_name'].fillna('') + ' ' + valid_seats['object_type'].fillna('')
coffee_term_pattern = r'\b(?:CAFÉ|CAFE(?:TERIA)?|COFFEE|ESPRESSO|TEA|BOBA|MATCHA|LATTE|ROASTERS?)\b'
coffee_name_signal = name_type.str.contains(coffee_term_pattern, case=False, regex=True, na=False)
name_review_mask = chain_true & ~cafe_type & coffee_name_signal

# A tabela agrupa nomes exatamente como registados; variantes de grafia permanecem separadas.
cafe_name_rows = valid_seats[cafe_network_mask | name_review_mask].copy()
cafe_name_rows['match_basis'] = np.where(cafe_name_rows['object_type'].eq('Café'),
                                         'Tipo Café — comparável principal',
                                         'Termo no nome/tipo — rever categoria')
print('Tabela: 30 nomes exatos com mais estabelecimentos; variantes de grafia não são fundidas.')
cafe_name_table = (cafe_name_rows.groupby(['match_basis','object_type','object_name'])['number']
    .agg(estabelecimentos='count', lugares_min='min', lugares_mediana='median', lugares_max='max')
    .reset_index().sort_values(['estabelecimentos','object_name'], ascending=[False,True])
    .set_index(['match_basis','object_type','object_name']).round(1))
print(f'Comparáveis principais: {int(cafe_network_mask.sum()):,} locais em {valid_seats.loc[cafe_network_mask, "object_name"].nunique():,} nomes exatos.')
print(f'Correspondências adicionais pelo nome/tipo, fora da categoria Café: {int(name_review_mask.sum()):,}.')
display(cafe_name_table.head(30))

# Apresentar a amplitude observada e o intervalo central da coorte de comparação principal.
cafe_network = valid_seats.loc[cafe_network_mask].copy()
cafe_seat_min = int(cafe_network['number'].min())
cafe_seat_max = int(cafe_network['number'].max())
cafe_q25 = float(cafe_network['number'].quantile(.25))
cafe_median = float(cafe_network['number'].median())
cafe_q75 = float(cafe_network['number'].quantile(.75))
cafe_capacity_reference = pd.DataFrame([{
    'Estabelecimentos':len(cafe_network), 'Mínimo observado':cafe_seat_min,
    'P25':cafe_q25, 'Mediana':cafe_median, 'P75':cafe_q75,
    'Máximo observado':cafe_seat_max
}]).round(1)
display(cafe_capacity_reference)
display(Markdown(f'**Leitura para a decisão.** Entre as cafetarias de rede identificadas, a capacidade observada vai de {cafe_seat_min} a {cafe_seat_max} lugares; a metade central situa-se entre {cafe_q25:.0f} e {cafe_q75:.0f}, com mediana de {cafe_median:.0f}. O mínimo e o máximo são extremos observados, não requisitos comerciais. Use a faixa central como referência inicial de teste, não como limite rígido.'))

### Capacidade comparável não significa viabilidade comprovada

O gráfico separa as cafetarias de rede observadas, os estabelecimentos de outras categorias cujo nome sugere uma oferta de café e os restantes locais. Nos restantes, a cor distingue capacidades dentro ou fora do intervalo central (P25–P75) das cafetarias de rede. Estar dentro indica semelhança de lugares; estar fora indica um formato menos habitual, não uma impossibilidade. A associação estatística serve para testar se a capacidade ajuda a distinguir os comparáveis, não para provar procura ou rentabilidade.

In [ ]:
comparison = valid_seats.copy()
comparison['is_cafe_network'] = cafe_network_mask.fillna(False).astype(bool)
comparison['is_name_review'] = name_review_mask.fillna(False).astype(bool)
comparison['chain_display'] = (comparison['chain'].astype(object)
    .map({True:'Rede',False:'Independente'}).fillna('Desconhecida').astype(str))
within_central_range = comparison['number'].ge(cafe_q25) & comparison['number'].le(cafe_q75)
comparison['comparison_group'] = 'Outro local — fora da faixa central'
comparison.loc[within_central_range.fillna(False), 'comparison_group'] = 'Outro local — capacidade comparável'
comparison.loc[comparison['is_name_review'], 'comparison_group'] = 'Nome sugere café — rever tipo'
comparison.loc[comparison['is_cafe_network'], 'comparison_group'] = 'Cafetaria de rede observada'

comparison_summary = (comparison.groupby('comparison_group')['number']
    .agg(estabelecimentos='count', mediana_lugares='median', minimo='min', maximo='max')
    .sort_values('estabelecimentos', ascending=False).round(1))
display(comparison_summary)

# O coeficiente de Spearman mede a associação entre o número de lugares e a pertença à coorte confirmada.
seat_network_rho = comparison['number'].corr(comparison['is_cafe_network'].astype(int), method='spearman')
other_places = comparison.loc[~comparison['is_cafe_network']]
share_other_in_central = 100 * (other_places['number'].ge(cafe_q25) & other_places['number'].le(cafe_q75)).mean()

type_order = ['Restaurante','Fast food','Café','Pizzaria','Bar','Padaria']
type_position = {name:position for position,name in enumerate(type_order)}
comparison['type_position'] = comparison['object_type'].map(type_position).astype(float)
# Small fixed-seed offsets separate overlapping observations and remain identical on each run.
comparison['plot_y'] = comparison['type_position'] + np.random.RandomState(42).uniform(
    -0.22, 0.22, size=len(comparison)
)

fig = px.scatter(
    comparison, x='number', y='plot_y', color='comparison_group',
    hover_name='object_name', hover_data={
        'number':':.0f', 'object_type':True, 'chain_display':True,
        'comparison_group':True, 'is_cafe_network':False, 'is_name_review':False,
        'type_position':False, 'plot_y':False
    },
    color_discrete_map={
        'Cafetaria de rede observada':'#197C78',
        'Nome sugere café — rever tipo':'#D98841',
        'Outro local — capacidade comparável':'#6A9FB5',
        'Outro local — fora da faixa central':'#C7CDD1'
    },
    title='Lugares por tipo: comparáveis de cafetarias de rede e restantes locais',
    labels={'number':'Lugares por estabelecimento','object_type':'Tipo de estabelecimento',
            'comparison_group':'Grupo de comparação','chain_display':'Classificação da rede'}
)
for trace in fig.data:
    if trace.name == 'Cafetaria de rede observada':
        trace.marker.update(size=9, opacity=.9, symbol='diamond')
    else:
        trace.marker.update(size=5, opacity=.35)
fig.update_yaxes(tickmode='array', tickvals=list(range(len(type_order))), ticktext=type_order,
                 range=[-.5,len(type_order)-.5], title='Tipo de estabelecimento')
fig.add_shape(type='rect', x0=cafe_q25, x1=cafe_q75, y0=-.5, y1=len(type_order)-.5,
              fillcolor='#F2C14E', opacity=.14, line=dict(width=0), layer='below')
fig.add_shape(type='line', x0=cafe_seat_min, x1=cafe_seat_min, y0=-.5, y1=len(type_order)-.5,
              line=dict(color='#197C78', dash='dash', width=1))
fig.add_shape(type='line', x0=cafe_seat_max, x1=cafe_seat_max, y0=-.5, y1=len(type_order)-.5,
              line=dict(color='#197C78', dash='dash', width=1))
fig.update_layout(template='plotly_white', height=600, legend_title_text='Grupo de comparação')
fig.show()

strength = 'muito fraca' if abs(seat_network_rho) < .10 else ('fraca' if abs(seat_network_rho) < .30 else 'moderada ou forte')
display(Markdown(f'**Leitura para a decisão.** {share_other_in_central:.1f}% dos outros estabelecimentos têm capacidade dentro da faixa central das redes de cafés. A associação entre número de lugares e pertença à coorte confirmada é {strength} (Spearman ρ = {seat_network_rho:.3f}); por isso, a capacidade, isoladamente, não identifica uma oportunidade real. Priorize a faixa central para dimensionar cenários, mas valide conceito, procura e economia no local. As correspondências pelo nome noutras categorias exigem revisão manual antes de serem tratadas como concorrentes diretos.'))

## 4. Capacidade de lugares e estrutura das redes

A média indica escala geral; a mediana e os quartis descrevem melhor o formato habitual quando existem estabelecimentos muito grandes. A comparação entre redes e independentes ajuda a avaliar se a expansão tende a depender de unidades compactas ou de espaços maiores.

In [ ]:
seat_summary = (valid_seats.groupby('object_type').number
    .agg(venues='count', mean='mean', median='median', q25=lambda s:s.quantile(.25), q75=lambda s:s.quantile(.75))
    .sort_values('mean', ascending=False).round(1))
display(seat_summary)
mean_ci = valid_seats.groupby('object_type').number.agg(['mean','std','count']).sort_values('mean')
mean_ci['ci95'] = 1.96 * mean_ci['std'] / np.sqrt(mean_ci['count'])
fig, ax = plt.subplots(figsize=(9,5.5))
ax.errorbar(mean_ci['mean'], mean_ci.index, xerr=mean_ci['ci95'], fmt='D',
            color='#197C78', ecolor='#6B8F8A', capsize=4, markersize=6)
ax.set(title='Média de lugares por tipo, com intervalo de confiança de 95%', xlabel='Lugares por estabelecimento', ylabel='')
sns.despine(); plt.tight_layout(); plt.show()
order = seat_summary.sort_values('median').index
fig, ax = plt.subplots(figsize=(9,5.5))
sns.boxplot(data=valid_seats, x='number', y='object_type', order=order, showfliers=False,
            color='#BBD8D4', width=.62, whis=(5,95), ax=ax)
ax.set(title='Distribuição dos lugares por tipo (caixa: intervalo interquartil; bigodes: P5–P95)',
       xlabel='Lugares por estabelecimento', ylabel='')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
chain_seats = classified[classified.number.notna() & classified.number.gt(0)].copy()
chain_capacity = chain_seats.groupby('chain').number.agg(venues='count', mean='mean', median='median', q25=lambda s:s.quantile(.25), q75=lambda s:s.quantile(.75)).round(1)
chain_capacity.index = chain_capacity.index.map({False:'Independente',True:'Rede'})
display(chain_capacity)
chain_seats['chain_status'] = np.where(chain_seats.chain, 'Rede', 'Independente')
fig, ax = plt.subplots(figsize=(8,4.8))
sns.boxplot(data=chain_seats, x='number', y='chain_status', order=['Independente','Rede'],
            hue='chain_status', hue_order=['Independente','Rede'], dodge=False,
            palette={'Independente':'#197C78','Rede':'#D98841'}, showfliers=False,
            width=.58, whis=(5,95), ax=ax)
if ax.get_legend() is not None: ax.get_legend().remove()
ax.set(title='Capacidade de lugares: redes e independentes', xlabel='Lugares por estabelecimento', ylabel='')
sns.despine(); plt.tight_layout(); plt.show()
display(Markdown(f'**Leitura para a decisão.** As redes têm mediana de {chain_capacity.loc["Rede","median"]:.0f} lugares, face a {chain_capacity.loc["Independente","median"]:.0f} nos independentes. O padrão observado é compatível com unidades de rede mais compactas; use esta referência para dimensionar o piloto, sem a tratar como prova de maior rentabilidade.'))

## 5. Verificação do paradoxo de Simpson: a composição altera a comparação

Comparamos a média de lugares nas redes e nos estabelecimentos independentes no conjunto total e dentro de cada tipo. O tipo é um factor plausível de composição, pois restaurantes, cafés e fast food têm distribuições de capacidade distintas. Um paradoxo de Simpson clássico exige que a direcção agregada se inverta de forma consistente nos grupos comparáveis; efeitos mistos entre tipos devem ser descritos como heterogeneidade, sem forçar essa classificação.


In [ ]:
# Comparar a média de lugares de redes e independentes no total e por tipo.
simpson_data = valid_seats[valid_seats['chain'].notna()].copy()
seat_by_type = (simpson_data.groupby(['object_type', 'chain'])['number']
    .agg(estabelecimentos='count', media_lugares='mean', mediana_lugares='median').unstack('chain'))
seat_by_type.columns = [f'{metric}_{"Rede" if status else "Independente"}'
                        for metric, status in seat_by_type.columns]
seat_by_type['diferenca_media_rede_menos_independente'] = (
    seat_by_type['media_lugares_Rede'] - seat_by_type['media_lugares_Independente'])
tipos_comparaveis = seat_by_type.dropna(subset=['media_lugares_Rede','media_lugares_Independente']).copy()

medias_total = simpson_data.groupby('chain')['number'].mean()
diferenca_total = float(medias_total.loc[True] - medias_total.loc[False])
amostra_comum = simpson_data[simpson_data['object_type'].isin(tipos_comparaveis.index)]
medias_comuns = amostra_comum.groupby('chain')['number'].mean()
diferenca_amostra_comum = float(medias_comuns.loc[True] - medias_comuns.loc[False])
diferenca_pesos_iguais = float(tipos_comparaveis['diferenca_media_rede_menos_independente'].mean())
diferencas_tipo = tipos_comparaveis['diferenca_media_rede_menos_independente']
inversao_estrita = bool((diferencas_tipo.gt(0).all() and diferenca_total < 0) or
                        (diferencas_tipo.lt(0).all() and diferenca_total > 0))

tabela_simpson = tipos_comparaveis[['estabelecimentos_Independente','estabelecimentos_Rede',
    'media_lugares_Independente','media_lugares_Rede',
    'diferenca_media_rede_menos_independente']].copy()
tabela_simpson = tabela_simpson.rename(columns={
    'estabelecimentos_Independente':'Estabelecimentos independentes',
    'estabelecimentos_Rede':'Estabelecimentos de rede',
    'media_lugares_Independente':'Média independentes',
    'media_lugares_Rede':'Média redes',
    'diferenca_media_rede_menos_independente':'Rede menos independente'})
display(tabela_simpson.round(1))
print(f'Todos os tipos agregados: redes {medias_total.loc[True]:.2f}, independentes {medias_total.loc[False]:.2f}; diferença {diferenca_total:+.2f} lugares.')
print(f'Comparação nas categorias com ambas as classificações ({len(tipos_comparaveis)} tipos): diferença {diferenca_amostra_comum:+.2f} lugares.')
print(f'Diferença padronizada com pesos iguais por tipo: {diferenca_pesos_iguais:+.2f} lugares (análise de sensibilidade, não composição preferencial do mercado).')

# As médias emparelhadas mostram a direcção da diferença em cada tipo; não há padarias independentes.
dados_grafico = tipos_comparaveis.sort_values('media_lugares_Independente')
fig, ax = plt.subplots(figsize=(9, 5.2))
for y, (tipo, linha) in enumerate(dados_grafico.iterrows()):
    media_ind, media_rede = linha['media_lugares_Independente'], linha['media_lugares_Rede']
    ax.plot([media_ind, media_rede], [y, y], color='#BECBC8', linewidth=2, zorder=1)
    ax.scatter(media_ind, y, color='#197C78', s=72, zorder=2)
    ax.scatter(media_rede, y, color='#D98841', s=72, zorder=2)
ax.set_yticks(list(range(len(dados_grafico))))
ax.set_yticklabels(dados_grafico.index.tolist())
ax.set(title='Média de lugares por tipo e classificação de rede', xlabel='Média de lugares por estabelecimento', ylabel='')
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0],marker='o',color='w',label='Independente',markerfacecolor='#197C78',markersize=8),
                   Line2D([0],[0],marker='o',color='w',label='Rede',markerfacecolor='#D98841',markersize=8)],
          title='Classificação', loc='lower right')
sns.despine(); plt.tight_layout(); plt.show()

sinais_tipo = int(diferencas_tipo.gt(0).sum()), int(diferencas_tipo.lt(0).sum())
resultado = ('Existe uma inversão estrita compatível com o paradoxo de Simpson.' if inversao_estrita else
             'Não se observa uma inversão estrita do paradoxo de Simpson: as diferenças por tipo têm direcções mistas.')
display(Markdown(f'**Leitura para a decisão.** {resultado} A diferença agregada entre redes e independentes é {diferenca_total:+.1f} lugares; com pesos iguais por tipo, é {diferenca_pesos_iguais:+.1f}, praticamente nula. As redes têm média superior em {sinais_tipo[0]} tipos comparáveis e inferior em {sinais_tipo[1]}; as padarias ficam de fora por não haver padarias independentes nesta base. O resultado é sensível à composição e heterogéneo; a média agregada não sustenta a conclusão geral de que redes precisam de menos lugares ou têm melhor desempenho.'))

## 6. Localização e capacidade

As dez ruas com mais estabelecimentos mostram onde a oferta está concentrada; o gráfico interativo relaciona concentração e mediana de lugares nas ruas com pelo menos 20 registos válidos. Use-o para selecionar corredores a investigar, não para escolher um local sem validar renda, fluxo pedonal e perfil de clientes.

In [ ]:
UNIT_RE = re.compile(r'\s+(?:#|APT\b|STE\b|SUITE\b|UNIT\b).*$', re.I)
def extract_street(address):
    if pd.isna(address): return pd.NA
    text = UNIT_RE.sub('', str(address).upper().strip())
    text = re.sub(r'^\s*\d+[A-Z]?\s+', '', text)
    return text.strip(' ,') or pd.NA

valid_type['street'] = valid_type.address.map(extract_street).astype('string')
street_counts = valid_type.dropna(subset=['street']).groupby('street').size().sort_values(ascending=False)
top10_streets = street_counts.head(10).rename('venues').reset_index()
single_restaurant_streets = int((street_counts == 1).sum())
print(f'Ruas com exatamente um estabelecimento: {single_restaurant_streets:,}')
display(top10_streets)
fig, ax = plt.subplots(figsize=(9,5.5))
sns.scatterplot(data=top10_streets, x='venues', y='street', size='venues', sizes=(80,220),
                color='#197C78', legend=False, ax=ax)
for _, row in top10_streets.iterrows():
    ax.annotate(f"{row.venues:,}", (row.venues, row.street), xytext=(8,0),
                textcoords='offset points', va='center', fontsize=8)
ax.set(title='Dez ruas com mais estabelecimentos', xlabel='Número de estabelecimentos', ylabel='')
sns.despine(); plt.tight_layout(); plt.show()
display(Markdown(f'**Leitura para a decisão.** {top10_streets.iloc[0]["street"]} lidera com {top10_streets.iloc[0]["venues"]:,} estabelecimentos. As {single_restaurant_streets:,} ruas com um único registo descrevem uma oferta dispersa, não uma oportunidade comprovada. Use a concentração para escolher locais a investigar e valide procura, renda e concorrência ao nível do corredor.'))

In [ ]:
busy_streets = street_counts.head(15).index
busy = valid_type[valid_type.street.isin(busy_streets) & valid_type.number.notna() & valid_type.number.gt(0)]
street_order = busy.groupby('street').number.median().sort_values().index.tolist()
fig = px.box(busy, x='number', y='street', category_orders={'street':street_order}, points='outliers',
             color='street', title='Distribuição interativa dos lugares nas 15 ruas mais concentradas',
             labels={'number':'Lugares por estabelecimento','street':'Rua'},
             hover_data={'street':False,'number':':.0f'})
fig.update_layout(template='plotly_white', showlegend=False, yaxis_title='', boxmode='group')
fig.show()
street_capacity = busy.groupby('street').number.agg(venues='count', median='median', mean='mean', q25=lambda s:s.quantile(.25), q75=lambda s:s.quantile(.75)).sort_values('venues',ascending=False).round(1)
display(street_capacity.head(15))
street_summary = (valid_type.dropna(subset=['street','number']).query('number > 0')
    .groupby('street').number.agg(venues='count', median='median', mean='mean', q25=lambda s:s.quantile(.25), q75=lambda s:s.quantile(.75)).reset_index())
street_summary = street_summary[street_summary.venues >= 20]
fig = px.scatter(street_summary, x='venues', y='median', size='venues', color='median',
                 hover_name='street', hover_data={'venues':True,'mean':':.1f','q25':':.1f','q75':':.1f'},
                 color_continuous_scale='Tealgrn', size_max=30,
                 title='Concentração de estabelecimentos e capacidade mediana por rua',
                 labels={'venues':'Estabelecimentos com lugares válidos','median':'Mediana de lugares'})
fig.update_layout(template='plotly_white', coloraxis_colorbar_title='Mediana')
fig.show()
display(Markdown('**Interpretação.** As ruas com maior concentração apresentam diferentes capacidades. No gráfico interactivo, passe o cursor sobre os pontos para consultar a rua, o número de estabelecimentos e os quartis; use o zoom para comparar zonas de maior concentração. A associação observada não demonstra que a concentração cause maior ou menor capacidade.'))

## 7. Verificação de interesse de pesquisa: Yandex Wordstat

**Estado do acesso:** a página pública do Wordstat explica o serviço, mas os resultados das consultas exigem uma conta autenticada. Nesta revisão não foi possível consultar volumes de pesquisa de Los Angeles; por isso, o relatório não apresenta estimativas de procura como se fossem medidas. O guia oficial descreve a evolução mensal, a distribuição por regiões/cidades e as consultas relacionadas; a API também requer um token OAuth. [Guia da interface Wordstat](https://yandex.com/support2/wordstat/en/interface/new) · [Requisitos de acesso à API](https://yandex.com/support2/wordstat/en/content/api-wordstat).

**Consultas para a próxima validação:** “coffee shop Los Angeles”, “cafe Los Angeles”, “robot cafe Los Angeles” e “robot restaurant Los Angeles”. Quando disponível, seleccionar Los Angeles/Califórnia, consultar a evolução mensal dos últimos dois anos e rever as consultas relacionadas. Registar data, expressão exacta, geografia, contagem, percentagem de consultas e índice de afinidade.

Interpretar o Wordstat apenas como sinal das pesquisas feitas no Yandex. Não representa toda a procura nos EUA, fluxo pedonal, disposição para pagar nem prova de viabilidade. Cruzar os resultados com entrevistas locais, fluxo nos corredores, visitas à concorrência e um piloto. Os resultados quantitativos ficam pendentes de uma exportação autenticada ou capturas de ecrã.


## 8. Síntese para o investidor

- **Estrutura competitiva:** restaurantes representam 75,2% do inventário; cafés, 4,5%. Redes representam 38,1% dos estabelecimentos com classificação conhecida. Há modelos replicáveis no mercado, mas a categoria café é menor e não constitui, por si só, prova de procura não atendida.
- **Referência de capacidade:** foram identificadas 266 cafetarias de rede pela classificação do tipo, com 175 nomes exatos. A mediana é 22 lugares; a metade central situa-se entre 12 e 37. O intervalo observado completo (1–143) é demasiado amplo para definir sozinho o formato.
- **Poder discriminatório dos lugares:** 41,8% dos restantes estabelecimentos estão dentro do intervalo central das cafetarias de rede; a associação entre lugares e pertença a essa coorte é muito fraca (Spearman ρ = −0,045). A capacidade ajuda a construir cenários, mas não identifica uma oportunidade real nem a sua rentabilidade.
- **Localização:** Wilshire Boulevard, W Sunset Boulevard e W Pico Boulevard lideram em número de estabelecimentos. A concentração pode refletir atividade e concorrência; as 574 ruas com um único registo não são automaticamente espaços de mercado por explorar.

**Implicação:** a tese de investimento depende de evidência local e operacional que este inventário não contém.

In [ ]:
# Estrutura de cenários; não é uma recomendação final de lugares.
scenario_table = pd.DataFrame([
    {'Cenário':'Piloto compacto','Capacidade ilustrativa':'Abaixo da mediana local de cafés','Escolha do atendimento':'Atendimento por robôs, com ajuda humana disponível','Objetivo':'Testar o fluxo com robôs, a qualidade do serviço e o atendimento nos períodos de maior afluência com menor exposição a custos fixos','Critério de decisão':'Expandir somente se regresso dos clientes, utilização e economia de pessoal atingirem as metas do piloto'},
    {'Cenário':'Piloto intermédio','Capacidade ilustrativa':'Próximo da mediana local de cafés','Escolha do atendimento':'Atendimento híbrido: o cliente escolhe assistência do robô ou da equipa','Objetivo':'Equilibrar flexibilidade de lugares com uma complexidade operacional administrável','Critério de decisão':'Validar clientes por assento, filas e transições entre robôs e equipa'},
    {'Cenário':'Formato amplo','Capacidade ilustrativa':'Acima da mediana local de cafés','Escolha do atendimento':'O cliente escolhe atendimento por robô, híbrido ou pela equipa','Objetivo':'Testar ocasiões em grupo e maior capacidade nos períodos de maior afluência','Critério de decisão':'Avançar somente com evidências de procura do local e confiabilidade robusta dos robôs'}
])
display(scenario_table)
display(Markdown('**Recomendação.** Trate esses formatos como hipóteses a testar, não como uma recomendação precisa de lugares. Use a mediana e os quartis das cafés locais como referência após escolher a rua e o local. Preserve a escolha do cliente em cada cenário: ofereça uma alternativa humana simples no piloto compacto, escolha clara entre robô e equipa no piloto híbrido e os três modos de serviço no teste amplo. Compare adesão, satisfação, acessibilidade, ocupação dos lugares por dia e horário, regresso dos clientes, tempo de serviço, intervenções nos robôs, custo de pessoal por cliente e margem de contribuição. Aprimore cada opção ao longo do tempo com base no feedback dos clientes e nos resultados operacionais. Uma estratégia de rede só é plausível quando o piloto demonstrar economia por unidade consistente e um modelo operacional que funcione sem depender da novidade.'))

## 9. Conclusão de investimento e próximos passos

### Decisão recomendada: financiar validação, não expansão
A evidência justifica um **piloto condicionado**, não um compromisso imediato com uma rede. Selecionar primeiro um corredor e validar procura, renda e concorrência. Depois testar cenários de capacidade em torno da referência central de 12–37 lugares, mantendo alternativas compactas e mais amplas para ocasiões diferentes. O mínimo observado de um lugar não é um mínimo comercial; o máximo de 143 não é uma meta recomendada.

Definir antes da abertura os limites de decisão para **regresso de clientes depois das promoções**, **ocupação por dia e horário**, **fiabilidade dos robôs**, **satisfação** e **margem de contribuição por unidade**. Só libertar capital para uma segunda localização se o piloto atingir as metas em semanas normais e o modelo continuar a funcionar sem depender da novidade.

### Diligências antes de comprometer capital
1. Comparar corredores candidatos por fluxo pedonal, renda, concorrência e procura de escritórios, habitação e transportes.
2. Testar os cenários compacto, intermédio e amplo; definir limites de ocupação, filas e retorno de clientes antes da abertura.
3. Medir preferência, satisfação e acessibilidade nos modos robótico, híbrido e humano.
4. Registar indisponibilidade, intervenções, manutenção e custo operacional dos robôs, incluindo a cobertura humana de contingência.
5. Construir a economia por unidade com rendas, pessoal, produtos, investimento, manutenção e financiamento; avançar apenas se a margem e a qualidade forem sustentáveis.